# Fine-tune the trip request parser (T29)

QLoRA on **Qwen2.5-3B-Instruct** to turn an RV trip request into the same JSON the Claude parser returns (`dave.parser.Extracted`). Training data: `training/data/train.jsonl` (T28).

**Run it on Colab:** Runtime → Change runtime type → **T4 GPU** (free tier is enough) → Runtime → Run all. Training takes roughly 30–45 minutes on a T4. Checkpoints and the adapter are saved to Google Drive under `MyDrive/dave/parser-qlora`.

Before you start, set `BRANCH` below to the branch to train from (`main` once the T28 and T29 PRs are merged). To compare with Claude, first run `uv run python -m dave.finetune baseline` on your machine (needs `ANTHROPIC_API_KEY`). It writes `training/results/baseline.json`. Commit it, or upload it to Drive at `MyDrive/dave/baseline.json`.

In [ ]:
REPO = "https://github.com/armita-jamshidi/RV-trip-agent"
BRANCH = "main"
BASE_MODEL = (
    "Qwen/Qwen2.5-3B-Instruct"  # switch to Qwen/Qwen2.5-7B-Instruct only if 3B misses the target
)
DRIVE_DIR = "/content/drive/MyDrive/dave/parser-qlora"

## 1. Setup

In [ ]:
%pip install -q transformers==4.55.4 trl==0.21.0 peft==0.17.1 bitsandbytes==0.47.0
%pip install -q accelerate==1.10.1 datasets==4.0.0 anthropic pydantic
!rm -rf /content/RV-trip-agent && git clone -q --depth 1 -b $BRANCH $REPO /content/RV-trip-agent

import sys

from google.colab import drive

sys.path.insert(0, "/content/RV-trip-agent/src")
drive.mount("/content/drive")

In [ ]:
from datasets import Dataset

from dave import finetune

train_rows, test_rows = finetune.load("train"), finetune.load("test")
train = Dataset.from_list([finetune.messages(r) for r in train_rows])
print(len(train_rows), "train,", len(test_rows), "test")
print(train[0]["prompt"][1]["content"])
print(train[0]["completion"][0]["content"])

## 2. Load the model in 4-bit and train a LoRA adapter

In [ ]:
import torch
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

bf16 = torch.cuda.is_bf16_supported()
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16 if bf16 else torch.float16,
    ),
    device_map="auto",
)
lora = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)
config = SFTConfig(
    output_dir=DRIVE_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    max_length=1024,
    completion_only_loss=True,  # learn the JSON answer, not the prompt
    gradient_checkpointing=True,
    bf16=bf16,
    fp16=not bf16,
    logging_steps=20,
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none",
    seed=29,
)
trainer = SFTTrainer(
    model=model, args=config, train_dataset=train, processing_class=tokenizer, peft_config=lora
)
trainer.train()
trainer.save_model(f"{DRIVE_DIR}/adapter")
tokenizer.save_pretrained(f"{DRIVE_DIR}/adapter")

## 3. Score on the held-out test set

In [ ]:
model = trainer.model
model.eval()
tokenizer.padding_side = "left"


def generate(rows, batch_size=16):
    texts = []
    for i in range(0, len(rows), batch_size):
        prompts = [
            tokenizer.apply_chat_template(
                finetune.messages(r)["prompt"], tokenize=False, add_generation_prompt=True
            )
            for r in rows[i : i + batch_size]
        ]
        batch = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)
        with torch.no_grad():
            out = model.generate(**batch, max_new_tokens=400, do_sample=False)
        texts += tokenizer.batch_decode(
            out[:, batch["input_ids"].shape[1] :], skip_special_tokens=True
        )
    return texts


texts = generate(test_rows)
outputs = [finetune.read_output(t) for t in texts]
report = finetune.score(outputs, test_rows)
print(f"valid JSON {report['valid_json']:.1%}, field accuracy {report['accuracy']:.1%}")
print({k: round(v, 3) for k, v in report["fields"].items()})

In [ ]:
import json
from pathlib import Path

report["model"], report["adapter"] = BASE_MODEL, f"{DRIVE_DIR}/adapter"
Path(f"{DRIVE_DIR}/finetuned.json").write_text(json.dumps(report, indent=2))

candidates = [
    Path("/content/RV-trip-agent/training/results/baseline.json"),
    Path("/content/drive/MyDrive/dave/baseline.json"),
]
base_path = next((p for p in candidates if p.exists()), None)
if base_path is None:
    print("No Claude baseline found; run `uv run python -m dave.finetune baseline` and upload it.")
else:
    base = json.loads(base_path.read_text())
    ratio = report["accuracy"] / base["accuracy"]
    verdict = "meets" if finetune.meets_target(report, base) else "misses"
    print(f"Claude {base['accuracy']:.1%} vs fine-tuned {report['accuracy']:.1%}:")
    print(f"{ratio:.1%} of the baseline, {verdict} the 95% target")

## 4. Publish the adapter (optional)

The adapter is already on Drive. To publish it on Hugging Face, add an `HF_TOKEN` secret in Colab (the key icon on the left), then run the cell below. Never paste the token into the notebook.

In [ ]:
from google.colab import userdata

HF_REPO = "armita-jamshidi/dave-parser-qwen2.5-3b-lora"  # change if you prefer another name
trainer.model.push_to_hub(HF_REPO, token=userdata.get("HF_TOKEN"), private=True)
tokenizer.push_to_hub(HF_REPO, token=userdata.get("HF_TOKEN"), private=True)

Then record the result in `context/decisions.md` (model, valid-JSON rate, field accuracy against the Claude baseline, where the adapter lives) and copy `finetuned.json` into `training/results/`.